<a href="https://colab.research.google.com/github/khovan123/cropstate/blob/main/notebooks/cropstate_colab_github_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CROPSTATE Colab Workflow

- Code clone từ GitHub; Dataset/KB/Results đọc-ghi trên Google Drive (`MyDrive/CROPSTATE_DATASET`, `CROPSTATE_KNOWLEDGE_BASE`, `CROPSTATE_RESULTS`).
- Dataset = **RiceSEG** (segmentation masks). Nhãn 6 stage suy từ mask, không có sẵn.
- Trước khi train: `Runtime > Change runtime type` → chọn **GPU**.

## Thứ tự chạy

**A. Chuẩn bị**
1. Mount Drive → 2. Clone/pull repo → 3. Cài deps → 4. Kiểm tra dataset/KB.

**B. Vision (RiceSEG, leak-free)**
5. **Cell 5** — gán nhãn 6 stage BBCH từ RiceSEG → `RICESEG_MANIFEST` (đọc `class_pixel_counts.xlsx`, suy từ mask) + audit manifest.
6. **Train `vision_final`** — dùng `--manifest {RICESEG_MANIFEST}`; split leak-free + balanced sampler tự áp. Rồi fine-tune, xem output, predict ảnh upload.
7. **Novelty** — retrain ordinal/focal + fixed-split multi-seed, bảng so sánh, phân tích post-hoc (calibration/temporal/leakage).

**C. Retrieval / Knowledge base** (độc lập với vision)
8. Audit corpus IRRI → run/evaluate retrieval (2 kịch bản minh hoạ) → retrieval đóng vòng từ belief thật (robust).

> ⚠️ Chuỗi layout cũ (folder `01_..06_`, encode parent, build/convert image manifest) **đã gỡ** — dataset giờ là RiceSEG, nhãn stage do cell 5 sinh. "Run all" an toàn.


In [1]:
# Sửa REPO_URL thành GitHub repo thật của bạn.
REPO_URL = "https://github.com/khovan123/cropstate"
PROJECT_DIR = "/content/CROPSTATE_Full_Research_Package"

DATA_ROOT = "/content/drive/MyDrive/CROPSTATE_DATASET"
KNOWLEDGE_ROOT = "/content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE"
RESULTS_ROOT = "/content/drive/MyDrive/CROPSTATE_RESULTS"
OUTPUT_DIR = f"{RESULTS_ROOT}/vision_final"

# Knowledge base: CHỈ dùng corpus IRRI (build từ raw_sources_irri/). Không dùng
# raw_sources hỗn hợp (PDF tiếng Việt/English) hay chunking cũ nữa.
KB_COMPLETE_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_irri_en.jsonl"
KB_NONRESTRICTED_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_irri_en_nonrestricted.jsonl"
RETRIEVAL_SCENARIOS = "data/retrieval_scenarios.csv"
# Dataset = RiceSEG (segmentation masks) -> gán nhãn 6 stage vào manifest này.
RICESEG_MANIFEST = "data/riceseg_stage_manifest.csv"

print("REPO_URL:", REPO_URL)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("KNOWLEDGE_ROOT:", KNOWLEDGE_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("RICESEG_MANIFEST:", RICESEG_MANIFEST)

REPO_URL: https://github.com/khovan123/cropstate
PROJECT_DIR: /content/CROPSTATE_Full_Research_Package
DATA_ROOT: /content/drive/MyDrive/CROPSTATE_DATASET
KNOWLEDGE_ROOT: /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE
RESULTS_ROOT: /content/drive/MyDrive/CROPSTATE_RESULTS
RICESEG_MANIFEST: data/riceseg_stage_manifest.csv


## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Clone hoặc update repo từ GitHub

In [3]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/CROPSTATE_Full_Research_Package


## 3. Cài dependencies

In [4]:
!pip install -r requirements.txt
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 85.2 MB/s eta 0:00:00
Obtaining file:///content/CROPSTATE_Full_Research_Package
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cropstate-research (pyproject.toml) ... done
  Created wheel for cropstate-research: filename=cropstate_research-0.1.0-0.editable-py3-none-any.whl size=1383 sha256=afbd8b9107c997ba87f66d4f3f3c855d25664668abc93f260c7b0e6742f42c94
  Stored in directory: /tmp/pip-ephem-wheel-cache-ikblci5r/wheels/37/37/8f/89550734e79d9e7fc68e041f688e3bc8f62ff3b93e9d2c762c
Successfully built cropstate-research


## 4. Kiểm tra dataset và knowledge base trên Drive

In [5]:
!ls -lah "{DATA_ROOT}"
!ls -lah "{KNOWLEDGE_ROOT}"

ls: /content/drive/MyDrive/CROPSTATE_DATASET: No such file or directory
lrw------- 1 root root 0 Jul 11 08:56 /content/drive/MyDrive/CROPSTATE_DATASET -> /content/drive/.shortcut-targets-by-id/1iazxjNfESn9lm7fn-1L8ETOAtKSWVDYN/CROPSTATE_DATASET
ls: /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE: No such file or directory
lrw------- 1 root root 0 Jul 11 09:02 /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE -> /content/drive/.shortcut-targets-by-id/1rwtO-iHapZuwjFhRqwYcXm9RTyrvuHqS/CROPSTATE_KNOWLEDGE_BASE


Dataset trên Drive giờ là **RiceSEG** (không còn folder `01_..06_`):

```text
CROPSTATE_DATASET/
  class_pixel_counts.xlsx      # pixel 6 lớp / mask (bg,green,senescent,panicle,weeds,duckweed)
  crop_mapping.xlsx
  segmentation/<nước>/<vùng>/label/*.png   # mask index 0-5
                          /rgb/*.jpg     # ảnh RGB tương ứng
```

Nhãn 6 stage được **suy từ mask** ở cell kế (không có sẵn trong dataset).

Knowledge base **chỉ dùng corpus IRRI**: `chunks/rice_knowledge_irri_en.jsonl` + `..._nonrestricted.jsonl` + `raw_sources_irri/`.


## 5. Gán nhãn 6 stage BBCH từ RiceSEG

Đọc `class_pixel_counts.xlsx`, suy giai đoạn từ hình thái trong mask (**bông** = panicle, **độ úa** = senescent, **độ phủ** = cover) → manifest 6 lớp `RICESEG_MANIFEST`, tương thích thẳng `train_vision.py` (leak-free split + balanced sampler tự áp).

- 3 lớp muộn (reproductive/grain_filling/ripening): dựa panicle+senescence → **chắc**.
- 3 lớp sớm (establishment/tillering/stem_booting): dựa cover → **yếu**; cột `review_flag` đánh dấu ca sát ranh giới. Chỉnh `--cover-low/-high`, `--senescent-low/-high` nếu lệch.


In [6]:
# RiceSEG -> manifest 6 stage (image_path tương đối DATA_ROOT, split=unassigned).
!PYTHONPATH=src python scripts/experiments/relabel_riceseg_bbch.py \
  --dataset-root "{DATA_ROOT}" \
  --output {RICESEG_MANIFEST}

# Kiểm tra manifest (cột, phân bố, leakage, file thiếu)
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest {RICESEG_MANIFEST} --data-root "{DATA_ROOT}"

RiceSEG rows: 3078 | unresolved rgb: 100 | dropped near-empty: 78 | labelled: 2900
Wrote data/riceseg_stage_manifest.csv

Stage distribution (review-flagged in parentheses):
  establishment    727  (208 to review)
  tillering        852  (417 to review)
  stem_booting     794  (194 to review)
  reproductive     288  (222 to review)
  grain_filling    110  (84 to review)
  ripening         129  (9 to review)
  parents (leak-free groups): 742
  flagged for review: 1134/2900
Rows: 2900

Class counts:
 macro_stage
tillering        852
stem_booting     794
establishment    727
reproductive     288
ripening         129
grain_filling    110
Name: count, dtype: int64

Split counts:
 split
unassigned    2900
Name: count, dtype: int64

parent_image_id leakage groups: 0

field_id leakage groups: 0

capture_session leakage groups: 0

Unlabeled rows: 0

Missing image files: 0

Non-training labels in manifest: 0


## 9. Audit corpus IRRI (complete)

Audit corpus IRRI đầy đủ (`rice_knowledge_irri_en.jsonl`) để xem coverage, restricted chunks và trạng thái production. Output lưu vào Drive results.

In [7]:
!mkdir -p "{RESULTS_ROOT}/retrieval"
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_COMPLETE_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_complete.json"


{
  "total_chunks": 173,
  "by_topic": {
    "disease_risk": 11,
    "general_crop_care": 16,
    "harvest_readiness": 18,
    "nutrient_management": 8,
    "pest_risk": 60,
    "residue_management": 16,
    "water_management": 16,
    "weed_management": 28
  },
  "by_facet": {
    "conditions": 48,
    "fertilizer": 10,
    "general": 73,
    "next_stage_action": 7,
    "pest_disease_prevention": 35
  },
  "by_source": {
    "IRRI_RKB_001": 8,
    "IRRI_RKB_002": 12,
    "IRRI_RKB_003": 5,
    "IRRI_RKB_004": 18,
    "IRRI_RKB_005": 8,
    "IRRI_RKB_006": 6,
    "IRRI_RKB_007": 25,
    "IRRI_RKB_008": 22,
    "IRRI_RKB_009": 17,
    "IRRI_RKB_010": 13,
    "IRRI_RKB_011": 9,
    "IRRI_RKB_012": 10,
    "IRRI_RKB_013": 11,
    "IRRI_RKB_014": 9
  },
  "by_review_status": {
    "machine_curated_pending_domain_review": 173
  },
  "stage_high_compatibility": {
    "establishment": 83,
    "tillering": 4,
    "stem_booting": 2,
    "reproductive": 5,
    "grain_filling": 1,
    "ripening":

## 10. Audit corpus IRRI (non-restricted)

Corpus IRRI đã loại chunk có restricted action (`rice_knowledge_irri_en_nonrestricted.jsonl`), phù hợp cho retrieval pilot.

In [8]:
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_NONRESTRICTED_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_nonrestricted.json"


{
  "total_chunks": 173,
  "by_topic": {
    "disease_risk": 11,
    "general_crop_care": 16,
    "harvest_readiness": 18,
    "nutrient_management": 8,
    "pest_risk": 60,
    "residue_management": 16,
    "water_management": 16,
    "weed_management": 28
  },
  "by_facet": {
    "conditions": 48,
    "fertilizer": 10,
    "general": 73,
    "next_stage_action": 7,
    "pest_disease_prevention": 35
  },
  "by_source": {
    "IRRI_RKB_001": 8,
    "IRRI_RKB_002": 12,
    "IRRI_RKB_003": 5,
    "IRRI_RKB_004": 18,
    "IRRI_RKB_005": 8,
    "IRRI_RKB_006": 6,
    "IRRI_RKB_007": 25,
    "IRRI_RKB_008": 22,
    "IRRI_RKB_009": 17,
    "IRRI_RKB_010": 13,
    "IRRI_RKB_011": 9,
    "IRRI_RKB_012": 10,
    "IRRI_RKB_013": 11,
    "IRRI_RKB_014": 9
  },
  "by_review_status": {
    "machine_curated_pending_domain_review": 173
  },
  "stage_high_compatibility": {
    "establishment": 83,
    "tillering": 4,
    "stem_booting": 2,
    "reproductive": 5,
    "grain_filling": 1,
    "ripening":

## 12. Run retrieval mẫu

Ví dụ: water management cho stage tillering. Script sẽ tải multilingual embedding model lần đầu chạy.

In [9]:
!PYTHONPATH=src python scripts/run_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --topic water_management \
  --stage tillering \
  --mode research \
  --top-k 5 \
  --output "{RESULTS_ROOT}/retrieval/water_tillering.json"


modules.json: 100% 229/229 [00:00<00:00, 1.16MB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 776kB/s]
README.md: 100% 3.89k/3.89k [00:00<00:00, 14.2MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 372kB/s]
config.json: 100% 645/645 [00:00<00:00, 2.98MB/s]
model.safetensors: 100% 471M/471M [00:02<00:00, 181MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 6393.12it/s]
tokenizer_config.json: 100% 526/526 [00:00<00:00, 3.30MB/s]
tokenizer.json: 100% 9.08M/9.08M [00:00<00:00, 15.0MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 1.53MB/s]
config.json: 100% 190/190 [00:00<00:00, 1.23MB/s]
{
  "topic": "water_management",
  "query": "Bằng chứng quản lý nước phù hợp với lúa ở giai đoạn Tillering, BBCH 20-29.",
  "belief": [
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0
  ],
  "confidence": 1.0,
  "mode": "research",
  "results": [
    {
      "rank": 1,
      "chunk_id": "IRRI_RKB_007_C0005",
      "text": "Large amounts of water can be lost dur

## 12b. Liệt kê chunk_id thật để tạo `data/retrieval_scenarios.csv`

Cell "Evaluate retrieval baselines" bên dưới cần file `data/retrieval_scenarios.csv` — đây là file bạn tự viết tay (ground truth đánh giá), không phải file do script tự sinh ra. Nếu file này chưa tồn tại, cell evaluate sẽ báo `FileNotFoundError`, đó là điều bình thường, không phải lỗi.

Cell này chỉ **liệt kê chunk_id thật** trong corpus của bạn theo từng topic/stage, để bạn có cơ sở chọn:

- `relevant_chunk_ids`: các chunk phù hợp giai đoạn đó (stage_compatibility ≥ 0.8);
- `incompatible_chunk_ids`: các chunk cùng topic nhưng không phù hợp giai đoạn đó (stage_compatibility = 0).

**Quan trọng:** danh sách này chỉ là gợi ý tự động dựa trên vector `stage_compatibility` sẵn có trong corpus. Không nên dùng thẳng làm ground truth khoa học cho paper, vì `stage_compatibility` cũng chính là tín hiệu mà bộ rerank dùng để xếp hạng — dùng lại nó làm nhãn đánh giá sẽ gây thiên lệch có lợi cho chính phương pháp đang được đánh giá (circular evaluation). Hãy dùng danh sách này làm điểm khởi đầu, sau đó tự rà soát/chỉnh sửa (hoặc nhờ người có chuyên môn nông nghiệp xác nhận) trước khi đưa vào `data/retrieval_scenarios.csv` thật.


In [10]:
import sys
sys.path.insert(0, "src")

from cropstate.knowledge import load_knowledge_chunks
from cropstate.constants import STAGE_NAMES

chunks = load_knowledge_chunks(KB_NONRESTRICTED_CORPUS, mode="research")
print(f"Total chunks loaded: {len(chunks)}\n")

topics = sorted({c.topic for c in chunks})
for topic in topics:
    topic_chunks = [c for c in chunks if c.topic == topic]
    print(f"=== topic: {topic} ({len(topic_chunks)} chunks) ===")
    for stage_index, stage in enumerate(STAGE_NAMES):
        relevant = [c for c in topic_chunks if c.stage_compatibility[stage_index] >= 0.5]
        incompatible = [c for c in topic_chunks if c.stage_compatibility[stage_index] == 0.0]
        if not relevant and not incompatible:
            continue
        print(f"  [{stage}]")
        if relevant:
            print(f"    relevant_chunk_ids candidates ({len(relevant)}):")
            for c in relevant[:8]:
                print(f"      {c.chunk_id}  -  {c.text[:70]}...")
        if incompatible:
            ids_preview = "|".join(c.chunk_id for c in incompatible[:8])
            print(f"    incompatible_chunk_ids candidates ({len(incompatible)}): {ids_preview}")
    print()

print("Copy các chunk_id phù hợp vào data/retrieval_scenarios.csv, cột relevant_chunk_ids")
print("và incompatible_chunk_ids cách nhau bằng dấu '|', ví dụ: kb_water_01|kb_water_07")


Total chunks loaded: 173

=== topic: disease_risk (11 chunks) ===
  [establishment]
    relevant_chunk_ids candidates (3):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
      IRRI_RKB_004_C0004  -  It must be grown, harvested, and processed correctly for best yield an...
      IRRI_RKB_004_C0017  -  The amount of moisture should be less than 14%, and preferably less th...
  [tillering]
    relevant_chunk_ids candidates (2):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
      IRRI_RKB_004_C0004  -  It must be grown, harvested, and processed correctly for best yield an...
    incompatible_chunk_ids candidates (1): IRRI_RKB_004_C0017
  [stem_booting]
    relevant_chunk_ids candidates (1):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
    incompatible_chunk_ids candidates (2): IRRI_RKB_004_C0004|IRRI_RKB_004_C0017
  [repr

## 13. Evaluate retrieval baselines

Chỉ chạy cell này khi đã có `data/retrieval_scenarios.csv`. Metrics gồm P@k, R@k, nDCG@k, SIRR@k cho ungated, hard top-1, fixed-soft, adaptive-soft và oracle.

In [11]:
!PYTHONPATH=src python scripts/evaluate_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --scenarios "{RETRIEVAL_SCENARIOS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/retrieval_evaluation.json"


Loading weights: 100% 199/199 [00:00<00:00, 4814.98it/s]
{
  "B0_ungated": {
    "precision_at_k": 0.1,
    "recall_at_k": 0.1,
    "ndcg_at_k": 0.06560253875617089,
    "sirr_at_k": 0.2
  },
  "B1_query_expansion": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.2
  },
  "B2_hard_filter": {
    "precision_at_k": 0.1,
    "recall_at_k": 0.16666666666666666,
    "ndcg_at_k": 0.11731968150568911,
    "sirr_at_k": 0.1
  },
  "B3_fixed_soft": {
    "precision_at_k": 0.2,
    "recall_at_k": 0.26666666666666666,
    "ndcg_at_k": 0.18292222026186,
    "sirr_at_k": 0.1
  },
  "P_adaptive_soft": {
    "precision_at_k": 0.1,
    "recall_at_k": 0.1,
    "ndcg_at_k": 0.06560253875617089,
    "sirr_at_k": 0.2
  },
  "oracle_reference": {
    "precision_at_k": 0.2,
    "recall_at_k": 0.26666666666666666,
    "ndcg_at_k": 0.30767353793273144,
    "sirr_at_k": 0.0
  }
}


## 19. Train lần đầu vào `vision_final`

Chỉ dùng một output folder duy nhất. Script tự tạo `OUTPUT_DIR/manifest.csv` từ các stage folder trong `DATA_ROOT`, rồi lưu checkpoint và test metrics vào cùng folder.

In [12]:
# Vision final trên RiceSEG. Manifest do cell 5 tạo; parent_image_id (nước/vùng/ảnh-gốc)
# đã group leak-free -> tắt hash (--leak-hamming-threshold 0) cho nhanh. python -u = log stream.
!PYTHONPATH=src python -u scripts/train_vision.py \
  --manifest {RICESEG_MANIFEST} \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --balanced-sampler \
  --leak-hamming-threshold 0 \
  --output "{OUTPUT_DIR}"

model.safetensors: 100% 46.8M/46.8M [00:00<00:00, 55.1MB/s]
1 {'accuracy': 0.06329113924050633, 'macro_f1': 0.045516569200779726, 'masd': 2.721518987341772, 'confusion_matrix': [[0, 0, 0, 2, 30, 12], [0, 0, 0, 0, 70, 20], [0, 0, 0, 0, 35, 22], [0, 0, 0, 0, 23, 2], [0, 0, 0, 0, 8, 1], [0, 0, 0, 0, 5, 7]], 'loss': 2.063643664121628, 'brier': 1.0036051239207462, 'ece': 0.2581793906064979, 'per_class_recall': {'establishment': 0.0, 'tillering': 0.0, 'stem_booting': 0.0, 'reproductive': 0.0, 'grain_filling': 0.8888888888888888, 'ripening': 0.5833333333333334}}
2 {'accuracy': 0.0759493670886076, 'macro_f1': 0.06439393939393939, 'masd': 2.6624472573839664, 'confusion_matrix': [[0, 0, 0, 0, 35, 9], [0, 0, 0, 0, 75, 15], [0, 0, 0, 0, 39, 18], [0, 0, 0, 0, 24, 1], [0, 0, 0, 0, 8, 1], [0, 0, 0, 0, 2, 10]], 'loss': 2.0468318313360214, 'brier': 1.0144316192343033, 'ece': 0.2717222527361117, 'per_class_recall': {'establishment': 0.0, 'tillering': 0.0, 'stem_booting': 0.0, 'reproductive': 0.0, 'grain

## 20. Fine-tune tiếp trong cùng `vision_final`

Dùng cell này cho lần chạy tiếp theo. Checkpoint cũ được đọc từ `OUTPUT_DIR/best_checkpoint.pt`; nếu validation tốt hơn, checkpoint mới ghi đè vào đúng file đó.

In [13]:
!PYTHONPATH=src python -u scripts/train_vision.py \
  --manifest "{OUTPUT_DIR}/manifest.csv" \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --resume-checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --freeze-backbone-epochs 0 \
  --learning-rate 0.0001 \
  --backbone-learning-rate 0.00001 \
  --output "{OUTPUT_DIR}"

31 {'accuracy': 0.7974683544303798, 'macro_f1': 0.765602725216387, 'masd': 0.24472573839662448, 'confusion_matrix': [[31, 13, 0, 0, 0, 0], [1, 73, 13, 1, 2, 0], [0, 4, 49, 0, 4, 0], [0, 0, 1, 19, 4, 1], [0, 0, 0, 2, 6, 1], [0, 0, 0, 0, 1, 11]], 'loss': 0.6362062245607376, 'brier': 0.35228247541804514, 'ece': 0.11562924898123439, 'per_class_recall': {'establishment': 0.7045454545454546, 'tillering': 0.8111111111111111, 'stem_booting': 0.8596491228070176, 'reproductive': 0.76, 'grain_filling': 0.6666666666666666, 'ripening': 0.9166666666666666}}
32 {'accuracy': 0.7805907172995781, 'macro_f1': 0.7592988151376711, 'masd': 0.25316455696202533, 'confusion_matrix': [[29, 15, 0, 0, 0, 0], [1, 71, 15, 1, 2, 0], [0, 6, 49, 0, 2, 0], [0, 0, 1, 19, 4, 1], [0, 0, 0, 2, 6, 1], [0, 0, 0, 0, 1, 11]], 'loss': 0.6074600829742849, 'brier': 0.33453399021606095, 'ece': 0.09066516060366411, 'per_class_recall': {'establishment': 0.6590909090909091, 'tillering': 0.7888888888888889, 'stem_booting': 0.859649122

## 21. Xem output đã lưu trên Drive

Folder này chỉ nên có `best_checkpoint.pt`, `history.json`, `class_counts.json`, `manifest.csv`, và `test_metrics.json`.

In [14]:
!ls -lah "{OUTPUT_DIR}"

total 44M
-rw------- 1 root root  43M Jul 11 10:27 best_checkpoint.pt
-rw------- 1 root root  477 Jul 11 10:31 class_counts.json
-rw------- 1 root root 140K Jul 11 10:31 history.json
-rw------- 1 root root 609K Jul 11 10:15 manifest.csv
-rw------- 1 root root 2.0K Jul 11 10:31 test_metrics.json


## 22. Test một ảnh upload từ máy

In [15]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
print("Uploaded:", image_path)

Saving cay-lua.png to cay-lua (1).png
Uploaded: cay-lua (1).png


In [16]:
!PYTHONPATH=src python scripts/predict_image.py \
  --checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --image "{image_path}"

{
  "image_path": "cay-lua (1).png",
  "predicted_id": 5,
  "predicted_stage": "ripening",
  "predicted_stage_display": "Ripening",
  "confidence": 0.3643556833267212,
  "stage_belief": {
    "establishment": 0.08335071802139282,
    "tillering": 0.2347043752670288,
    "stem_booting": 0.12550415098667145,
    "reproductive": 0.17242158949375153,
    "grain_filling": 0.019663505256175995,
    "ripening": 0.3643556833267212
  }
}


## 23. Novelty experiments — ordinal/focal loss + fixed-split multi-seed (GPU)

Các thí nghiệm tăng novelty cho paper. Tất cả retrain ResNet-18 trên **cùng một split cố định** với `vision_final` (`OUTPUT_DIR/manifest.csv`) để so sánh trực tiếp với baseline CE:

- **ordinal** (Tier A#2): loss = CE + λ·(E[stage] − stage_thật)² → phạt mạnh lỗi cách xa giai đoạn, giảm MASD và lỗi non-adjacent.
- **focal** (Tier C#6): giảm trọng số mẫu dễ, giúp lớp hiếm (reproductive).
- **seed 7 / seed 123** (Tier C#9): cùng split cố định, chỉ đổi seed → tách phương sai khởi tạo khỏi phương sai split.

**Trước khi chạy:**
1. Đã `git push` code mới lên GitHub: `scripts/train_vision.py` (thêm `--loss`), `src/cropstate/losses.py`, `scripts/experiments/`.
2. Đã chạy lại **cell 2 (git pull)** để Colab có code mới.
3. Đã chạy **cell 19** (train `vision_final`) để có split cố định.

Colab có GPU nên **không** cần `CROPSTATE_FORCE_CPU`. Mỗi lần train ~vài phút trên GPU.

In [17]:
NOVELTY_DIR = f"{RESULTS_ROOT}/novelty"
FIXED_MANIFEST = f"{OUTPUT_DIR}/manifest.csv"  # split cố định từ baseline CE (cell 19)

import os
os.makedirs(NOVELTY_DIR, exist_ok=True)
assert os.path.exists(FIXED_MANIFEST), (
    f"Thiếu {FIXED_MANIFEST} — chạy cell 19 (train vision_final) trước để tạo split cố định."
)
print("Fixed split :", FIXED_MANIFEST)
print("Novelty dir :", NOVELTY_DIR)

Fixed split : /content/drive/MyDrive/CROPSTATE_RESULTS/vision_final/manifest.csv
Novelty dir : /content/drive/MyDrive/CROPSTATE_RESULTS/novelty


In [18]:
# Tier A#2 — ordinal loss (CE + λ·(E[stage] − stage_thật)²)
!PYTHONPATH=src python -u scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss ordinal \
  --output "{NOVELTY_DIR}/resnet18_ordinal"

1 {'accuracy': 0.22784810126582278, 'macro_f1': 0.0928288068355203, 'masd': 1.40084388185654, 'confusion_matrix': [[38, 6, 0, 0, 0, 0], [74, 16, 0, 0, 0, 0], [43, 14, 0, 0, 0, 0], [15, 10, 0, 0, 0, 0], [5, 4, 0, 0, 0, 0], [7, 5, 0, 0, 0, 0]], 'loss': 2.55817049741745, 'brier': 0.7636889603173141, 'ece': 0.10023270547389986, 'per_class_recall': {'establishment': 0.8636363636363636, 'tillering': 0.17777777777777778, 'stem_booting': 0.0, 'reproductive': 0.0, 'grain_filling': 0.0, 'ripening': 0.0}}
2 {'accuracy': 0.20675105485232068, 'macro_f1': 0.11793797476362185, 'masd': 1.4641350210970465, 'confusion_matrix': [[42, 2, 0, 0, 0, 0], [86, 4, 0, 0, 0, 0], [50, 6, 1, 0, 0, 0], [19, 4, 1, 1, 0, 0], [6, 1, 0, 1, 1, 0], [11, 1, 0, 0, 0, 0]], 'loss': 2.3705853074789047, 'brier': 0.7443079533748718, 'ece': 0.19338821107325171, 'per_class_recall': {'establishment': 0.9545454545454546, 'tillering': 0.044444444444444446, 'stem_booting': 0.017543859649122806, 'reproductive': 0.04, 'grain_filling': 0

In [19]:
# Tier C#6 — focal loss (giúp lớp hiếm reproductive)
!PYTHONPATH=src python -u scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss focal \
  --output "{NOVELTY_DIR}/resnet18_focal"

1 {'accuracy': 0.27848101265822783, 'macro_f1': 0.2436745392441595, 'masd': 1.1687763713080168, 'confusion_matrix': [[10, 23, 0, 7, 4, 0], [1, 33, 0, 49, 7, 0], [1, 6, 0, 46, 4, 0], [0, 1, 1, 18, 4, 1], [0, 1, 0, 5, 3, 0], [0, 0, 0, 8, 2, 2]], 'loss': 1.4900940991938114, 'brier': 0.8076132412779512, 'ece': 0.07518106912761799, 'per_class_recall': {'establishment': 0.22727272727272727, 'tillering': 0.36666666666666664, 'stem_booting': 0.0, 'reproductive': 0.72, 'grain_filling': 0.3333333333333333, 'ripening': 0.16666666666666666}}
2 {'accuracy': 0.5569620253164557, 'macro_f1': 0.5040033145644303, 'masd': 0.6497890295358649, 'confusion_matrix': [[23, 16, 0, 1, 4, 0], [4, 71, 5, 3, 5, 2], [1, 27, 10, 12, 5, 2], [0, 1, 1, 13, 8, 2], [0, 1, 1, 1, 5, 1], [0, 0, 0, 0, 2, 10]], 'loss': 1.3943878076970577, 'brier': 0.780663430612122, 'ece': 0.3492044631690416, 'per_class_recall': {'establishment': 0.5227272727272727, 'tillering': 0.7888888888888889, 'stem_booting': 0.17543859649122806, 'reprodu

In [20]:
# Tier C#9 — fixed-split, seed 7
!PYTHONPATH=src python -u scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 7 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed7"

1 {'accuracy': 0.5021097046413502, 'macro_f1': 0.40769500486551863, 'masd': 0.6160337552742616, 'confusion_matrix': [[17, 23, 1, 1, 2, 0], [6, 75, 6, 3, 0, 0], [0, 46, 5, 4, 0, 2], [0, 2, 2, 16, 5, 0], [1, 1, 1, 3, 3, 0], [0, 0, 0, 4, 5, 3]], 'loss': 1.6756138652563095, 'brier': 0.7858955483848794, 'ece': 0.2947263445406524, 'per_class_recall': {'establishment': 0.38636363636363635, 'tillering': 0.8333333333333334, 'stem_booting': 0.08771929824561403, 'reproductive': 0.64, 'grain_filling': 0.3333333333333333, 'ripening': 0.25}}
2 {'accuracy': 0.5485232067510548, 'macro_f1': 0.4979543168506397, 'masd': 0.5063291139240507, 'confusion_matrix': [[22, 18, 3, 0, 1, 0], [5, 67, 16, 2, 0, 0], [0, 36, 18, 3, 0, 0], [0, 0, 5, 16, 4, 0], [0, 0, 2, 4, 3, 0], [0, 0, 0, 3, 5, 4]], 'loss': 1.5935302823781967, 'brier': 0.7524740801656966, 'ece': 0.32250185078206445, 'per_class_recall': {'establishment': 0.5, 'tillering': 0.7444444444444445, 'stem_booting': 0.3157894736842105, 'reproductive': 0.64, 'gr

In [21]:
# Tier C#9 — fixed-split, seed 123
!PYTHONPATH=src python -u scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 123 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed123"

1 {'accuracy': 0.4810126582278481, 'macro_f1': 0.3781371906075866, 'masd': 0.70042194092827, 'confusion_matrix': [[14, 26, 2, 2, 0, 0], [7, 72, 6, 4, 0, 1], [0, 45, 11, 1, 0, 0], [0, 10, 3, 11, 0, 1], [1, 2, 0, 4, 0, 2], [0, 3, 0, 3, 0, 6]], 'loss': 1.6694603115320206, 'brier': 0.7830932081060649, 'ece': 0.27217205755318263, 'per_class_recall': {'establishment': 0.3181818181818182, 'tillering': 0.8, 'stem_booting': 0.19298245614035087, 'reproductive': 0.44, 'grain_filling': 0.0, 'ripening': 0.5}}
2 {'accuracy': 0.6033755274261603, 'macro_f1': 0.502482683361983, 'masd': 0.4641350210970464, 'confusion_matrix': [[27, 16, 1, 0, 0, 0], [1, 80, 5, 3, 0, 1], [0, 44, 12, 1, 0, 0], [0, 1, 1, 14, 6, 3], [0, 2, 0, 3, 0, 4], [0, 0, 0, 1, 1, 10]], 'loss': 1.5961190164089203, 'brier': 0.7617780440562975, 'ece': 0.3815747408298501, 'per_class_recall': {'establishment': 0.6136363636363636, 'tillering': 0.8888888888888888, 'stem_booting': 0.21052631578947367, 'reproductive': 0.56, 'grain_filling': 0.0,

## 24. So sánh biến thể novelty với baseline CE

Đọc `test_metrics.json` của từng biến thể (cùng split cố định) và in bảng: accuracy, macro-F1, MASD, ECE, recall lớp reproductive. Kỳ vọng: **ordinal** giảm MASD và lỗi non-adjacent; **focal** tăng recall reproductive; **seed7/seed123** cho dải phương sai khởi tạo trên split cố định.

In [22]:
import json, os
import pandas as pd

variants = {
    "CE baseline (vision_final)": OUTPUT_DIR,
    "ordinal (A#2)":              f"{NOVELTY_DIR}/resnet18_ordinal",
    "focal (C#6)":                f"{NOVELTY_DIR}/resnet18_focal",
    "CE seed7 (C#9)":             f"{NOVELTY_DIR}/resnet18_ce_seed7",
    "CE seed123 (C#9)":           f"{NOVELTY_DIR}/resnet18_ce_seed123",
}
rows = []
for name, d in variants.items():
    p = os.path.join(d, "test_metrics.json")
    if not os.path.exists(p):
        print("bỏ qua (chưa train):", name)
        continue
    m = json.load(open(p))
    rows.append({
        "variant": name,
        "accuracy": round(m.get("accuracy", float("nan")), 3),
        "macro_f1": round(m.get("macro_f1", float("nan")), 3),
        "masd": round(m.get("masd", float("nan")), 3),
        "ece": round(m.get("ece", float("nan")), 3),
        "reproductive_recall": round((m.get("per_class_recall") or {}).get("reproductive", float("nan")), 3),
    })

df = pd.DataFrame(rows)
df.to_csv(f"{NOVELTY_DIR}/retrain_comparison.csv", index=False)
print(df.to_string(index=False))
print("\nĐã lưu:", f"{NOVELTY_DIR}/retrain_comparison.csv")

                   variant  accuracy  macro_f1  masd   ece  reproductive_recall
CE baseline (vision_final)     0.791     0.782 0.255 0.072                0.833
             ordinal (A#2)     0.810     0.788 0.224 0.067                0.917
               focal (C#6)     0.787     0.780 0.251 0.207                0.917
            CE seed7 (C#9)     0.817     0.787 0.205 0.048                0.792
          CE seed123 (C#9)     0.806     0.818 0.224 0.057                0.958

Đã lưu: /content/drive/MyDrive/CROPSTATE_RESULTS/novelty/retrain_comparison.csv


## 25. (Tuỳ chọn) Phân tích post-hoc trên Colab → lưu vào Drive

Export logits từ các checkpoint có sẵn rồi chạy calibration (temperature scaling, Tier A#3), temporal fusion (Tier B#5) và leakage audit (Tier A#1). Cần checkpoint `vision_final` (và nếu có `vision_small_cnn`, `vision_efficientnet_b0`) trong `RESULTS_ROOT`. Kết quả JSON lưu vào `RESULTS_ROOT/novelty` để kéo về.

In [23]:
CKPTS = {
    "resnet18": f"{OUTPUT_DIR}/best_checkpoint.pt",
    "small_cnn": f"{RESULTS_ROOT}/vision_small_cnn/best_checkpoint.pt",
    "efficientnet_b0": f"{RESULTS_ROOT}/vision_efficientnet_b0/best_checkpoint.pt",
}
import os
ck = " ".join(f'--checkpoint {n}={p}' for n, p in CKPTS.items() if os.path.exists(p))
print("Checkpoints dùng được:", ck or "(chỉ resnet18)")

!PYTHONPATH=src python scripts/experiments/export_logits.py {ck} \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --output-dir "{NOVELTY_DIR}/logits"

# Calibration (temperature scaling) — torch-free, cắt ECE
!PYTHONPATH=src python scripts/experiments/calibration.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" \
  --output "{NOVELTY_DIR}/calibration.json"

# Temporal phenology fusion (Tier B#5)
!PYTHONPATH=src python scripts/experiments/temporal_fusion.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" --model resnet18 \
  --output "{NOVELTY_DIR}/temporal_fusion.json"

# Near-duplicate / leakage audit (Tier A#1)
!PYTHONPATH=src python scripts/experiments/leakage_audit.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" --hamming-threshold 10 \
  --output "{NOVELTY_DIR}/leakage_audit.json" \
  --grouped-manifest "{NOVELTY_DIR}/manifest_hash_grouped.csv"

print("Xong. Kết quả trong", NOVELTY_DIR)

Checkpoints dùng được: --checkpoint resnet18=/content/drive/MyDrive/CROPSTATE_RESULTS/vision_final/best_checkpoint.pt
[export] resnet18: validation=237, test=263
[resnet18] T=0.871  ECE 0.072 -> 0.076   Brier 0.302 -> 0.300   NLL 0.540 -> 0.546
{
  "single_frame_argmax": {
    "accuracy": 0.7908745247148289,
    "masd": 0.25475285171102663,
    "non_adjacent_errors": 7,
    "adjacent_errors": 48
  },
  "temporal_fused_mean_over_20_orderings": {
    "accuracy": 0.8180608365019012,
    "masd": 0.18745247148288974,
    "non_adjacent_errors": 1.3,
    "adjacent_errors": 46.55
  },
  "non_adjacent_error_reduction": 5.7,
  "masd_reduction": 0.06730038022813689
}
Traceback (most recent call last):
  File "/content/CROPSTATE_Full_Research_Package/scripts/experiments/leakage_audit.py", line 142, in <module>
    main()
  File "/content/CROPSTATE_Full_Research_Package/scripts/experiments/leakage_audit.py", line 136, in main
    df.to_csv(args.grouped_manifest, index=False)
  File "/usr/local/lib/

## 12. Retrieval đóng vòng từ belief thật (robust)

`evaluate_retrieval` ở mục trước chỉ có 2 kịch bản minh hoạ. Ở đây sinh kịch bản
từ **belief 6 lớp thật của mọi ảnh test** (logits vừa export), gán relevance
tự động theo `stage_compatibility` của từng chunk IRRI (không cần chuyên gia gán tay).
Chỉ giữ cặp (topic, stage) mà corpus trả lời được (≥3 relevant & ≥3 incompatible),
nên số liệu không còn suy biến về 0 như bản 2-kịch-bản.


In [24]:
# Sinh kịch bản từ belief thật + corpus IRRI, rồi đánh giá retrieval (đóng vòng).
!PYTHONPATH=src python scripts/experiments/build_belief_scenarios.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" --model resnet18 \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --calibration "{NOVELTY_DIR}/calibration.json" \
  --output "{NOVELTY_DIR}/belief_scenarios.csv"

!PYTHONPATH=src python scripts/evaluate_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --scenarios "{NOVELTY_DIR}/belief_scenarios.csv" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/retrieval_evaluation_belief.json"

[scenarios] wrote 420 scenarios over 5 topics, 263 test images, temperature=0.871
Loading weights: 100% 199/199 [00:00<00:00, 7215.93it/s]
{
  "B0_ungated": {
    "precision_at_k": 0.22476190476190477,
    "recall_at_k": 0.11832548403976974,
    "ndcg_at_k": 0.16509145548789161,
    "sirr_at_k": 0.1357142857142857
  },
  "B1_query_expansion": {
    "precision_at_k": 0.37047619047619046,
    "recall_at_k": 0.19211919450641254,
    "ndcg_at_k": 0.33555434652637506,
    "sirr_at_k": 0.15857142857142856
  },
  "B2_hard_filter": {
    "precision_at_k": 0.26571428571428574,
    "recall_at_k": 0.1802588889806935,
    "ndcg_at_k": 0.25868191526355283,
    "sirr_at_k": 0.002380952380952381
  },
  "B3_fixed_soft": {
    "precision_at_k": 0.2604761904761905,
    "recall_at_k": 0.16271860971109092,
    "ndcg_at_k": 0.26548559212469836,
    "sirr_at_k": 0.07238095238095238
  },
  "P_adaptive_soft": {
    "precision_at_k": 0.3004761904761905,
    "recall_at_k": 0.19821359717976256,
    "ndcg_at_k": 

## Ghi chú

- GitHub chỉ lưu code và metadata nhỏ.
- Google Drive lưu dataset, knowledge base, checkpoint và kết quả training.
- Không commit các file `.pt` lên GitHub.